In [21]:
# creating spark session
exec(open('/home/jovyan/.ipython/profile_default/startup/00-spark-session.py').read())

Spark 3.5.0 session ready as `spark` (Delta Lake enabled).


# Immediate Food Delivery

## Problem Description

You are given a PySpark DataFrame named **`delivery`** containing food delivery orders.

An order is considered **immediate** when the customer's preferred delivery date is the same as the order date.

Calculate the **percentage of immediate orders out of all orders**.

The result should be rounded to **2 decimal places**.

### Input DataFrame

The `delivery` DataFrame contains:

| Column | Data Type | Description |
|---|---|---|
| `delivery_id` | Integer | Unique identifier for the delivery |
| `customer_id` | Integer | Identifier of the customer |
| `order_date` | String | Date when the order was placed |
| `customer_pref_delivery_date` | String | Customer's preferred delivery date |

Both dates are provided in `YYYY-MM-DD` format.

### Definition

An order is **immediate** when:

`order_date = customer_pref_delivery_date`

### Expected Output

Return a single column:

| Column | Description |
|---|---|
| `immediate_percentage` | Percentage of orders that are immediate |

Round the result to **2 decimal places**.

### Example

| delivery_id | customer_id | order_date | customer_pref_delivery_date |
|---:|---:|---|---|
| 1 | 101 | 2023-01-01 | 2023-01-01 |
| 2 | 102 | 2023-01-02 | 2023-01-05 |
| 3 | 103 | 2023-01-03 | 2023-01-03 |
| 4 | 104 | 2023-01-04 | 2023-01-08 |
| 5 | 105 | 2023-01-05 | 2023-01-05 |

### Expected Result

| immediate_percentage |
|---:|
| 60.00 |

### Calculation

There are **3 immediate orders** out of **5 total orders**.

`(3 / 5) × 100 = 60.00%`

### Problem Pattern

**Filtering / Conditional Aggregation → COUNT → Percentage → ROUND**

In [22]:

data = [
    (1, 101, "2023-01-01", "2023-01-01"),
    (2, 102, "2023-01-02", "2023-01-05"),
    (3, 103, "2023-01-03", "2023-01-03"),
    (4, 104, "2023-01-04", "2023-01-08"),
    (5, 105, "2023-01-05", "2023-01-05"),
    (6, 106, "2023-01-06", "2023-01-10"),
    (7, 107, "2023-01-07", "2023-01-07"),
    (8, 108, "2023-01-08", "2023-01-12"),
    (9, 109, "2023-01-09", "2023-01-09"),
    (10, 110, "2023-01-10", "2023-01-15")
]

columns = [
    "delivery_id",
    "customer_id",
    "order_date",
    "customer_pref_delivery_date"
]

delivery = spark.createDataFrame(data, columns)

delivery.show()

+-----------+-----------+----------+---------------------------+
|delivery_id|customer_id|order_date|customer_pref_delivery_date|
+-----------+-----------+----------+---------------------------+
|          1|        101|2023-01-01|                 2023-01-01|
|          2|        102|2023-01-02|                 2023-01-05|
|          3|        103|2023-01-03|                 2023-01-03|
|          4|        104|2023-01-04|                 2023-01-08|
|          5|        105|2023-01-05|                 2023-01-05|
|          6|        106|2023-01-06|                 2023-01-10|
|          7|        107|2023-01-07|                 2023-01-07|
|          8|        108|2023-01-08|                 2023-01-12|
|          9|        109|2023-01-09|                 2023-01-09|
|         10|        110|2023-01-10|                 2023-01-15|
+-----------+-----------+----------+---------------------------+



# Using Spark SQL

In [23]:
delivery.createOrReplaceTempView("delivery")

In [24]:
spark.sql("""
    SELECT
        ROUND(
            100.0 * SUM(
                CASE
                    WHEN order_date = customer_pref_delivery_date THEN 1
                    ELSE 0
                END
            ) / COUNT(*),
            2
        ) AS immediate_percentage
    FROM delivery
""").show()

+--------------------+
|immediate_percentage|
+--------------------+
|               50.00|
+--------------------+



# Pyspark

In [19]:
result = delivery.select(
    round(
        100.0 * avg(
            (col("order_date") == col("customer_pref_delivery_date")).cast("int")
        ),
        2
    ).alias("immediate_percentage")
)

result.show()

+--------------------+
|immediate_percentage|
+--------------------+
|                50.0|
+--------------------+

